# β-VAE Training — Credit Card Fraud Detection

## 1. Clone Repository & Install Dependencies

In [69]:
import os, subprocess, sys

REPO_URL = 'https://github.com/nanokwok/Deep-Fraud-VAE.git'
REPO_DIR = '/content/Deep-Fraud-VAE'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — consider enabling Runtime > Change runtime type > GPU')

Repo already cloned — pulling latest
PyTorch : 2.10.0+cpu
CUDA    : False
No GPU — consider enabling Runtime > Change runtime type > GPU


## 2. Download Dataset from Kaggle

In [70]:
import os, json

# TODO: Replace with your Kaggle credentials. Do NOT commit.
KAGGLE_USERNAME = 'username'
KAGGLE_KEY      = 'key'

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials saved.')

Kaggle credentials saved.


In [71]:
import os, subprocess, sys

RAW_DIR = 'data/raw'
RAW_CSV = f'{RAW_DIR}/creditcard.csv'
os.makedirs(RAW_DIR, exist_ok=True)

if os.path.exists(RAW_CSV):
    size_mb = os.path.getsize(RAW_CSV) / 1e6
    print(f'creditcard.csv already present ({size_mb:.1f} MB) — skipping download.')
else:
    print('Downloading Credit Card Fraud dataset from Kaggle (~144 MB)...')
    # Dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
    subprocess.run([
        'kaggle', 'datasets', 'download',
        'mlg-ulb/creditcardfraud',
        '-p', RAW_DIR, '--unzip',
    ], check=True)
    size_mb = os.path.getsize(RAW_CSV) / 1e6
    print(f'Download complete: {size_mb:.1f} MB')


creditcard.csv already present (150.8 MB) — skipping download.


In [72]:
import subprocess
import sys

# The same command that failed previously
command = [
    'kaggle', 'datasets', 'download',
    'mlg-ulb/creditcardfraud',
    '-p', 'data/raw', '--unzip'
]

# Execute the command without 'check=True' to prevent another CalledProcessError
# and print stdout/stderr explicitly.
print(f"Executing: {' '.join(command)}")
process = subprocess.run(command, capture_output=True, text=True)

print("\n--- STDOUT ---")
print(process.stdout)
print("\n--- STDERR ---")
print(process.stderr)

if process.returncode != 0:
    print(f"\nCommand failed with exit code: {process.returncode}")
else:
    print("\nCommand completed successfully.")


Executing: kaggle datasets download mlg-ulb/creditcardfraud -p data/raw --unzip

--- STDOUT ---
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0



--- STDERR ---

  0%|          | 0.00/66.0M [00:00<?, ?B/s]
 14%|█▎        | 9.00M/66.0M [00:00<00:00, 91.2MB/s]
 42%|████▏     | 28.0M/66.0M [00:00<00:00, 153MB/s] 
 79%|███████▉  | 52.0M/66.0M [00:00<00:00, 195MB/s]
100%|██████████| 66.0M/66.0M [00:00<00:00, 191MB/s]


Command completed successfully.


## 3. Run Preprocessing Pipeline

Runs `src/preprocess.py` to generate the `.npy` feature arrays.

Steps: load `creditcard.csv` → semi-supervised split (80% normal → train, 20% normal + 100% fraud → val) → `StandardScaler` on Time, `RobustScaler` on Amount, V1–V28 untouched → save.

Expected runtime: < 30 seconds.

In [73]:
import subprocess, sys, numpy as np, json
from pathlib import Path

PROC_DIR = Path('data/processed')
REQUIRED = ['X_train.npy', 'X_val.npy', 'y_train.npy', 'y_val.npy',
            'feature_columns.json', 'scaler.pkl']

if all((PROC_DIR / f).exists() for f in REQUIRED):
    print('Processed arrays already exist — skipping. Delete data/processed/ to rerun.')
else:
    print('Running preprocessing...')
    subprocess.run([sys.executable, '-m', 'src.preprocess'], check=True)

X_train = np.load(PROC_DIR / 'X_train.npy')
X_val   = np.load(PROC_DIR / 'X_val.npy')
y_val   = np.load(PROC_DIR / 'y_val.npy')
with open(PROC_DIR / 'feature_columns.json') as f:
    feat_names = json.load(f)

print(f'X_train : {X_train.shape}  dtype={X_train.dtype}  (normal only)')
print(f'X_val   : {X_val.shape}  dtype={X_val.dtype}')
print(f'y_val   : fraud_rate={y_val.mean()*100:.4f}%  fraud_count={y_val.sum()}')
print(f'features: {feat_names}')
assert not np.isnan(X_train).any(), 'NaN detected in X_train!'
assert not np.isnan(X_val).any(),   'NaN detected in X_val!'
print('NaN check passed.')

Processed arrays already exist — skipping. Delete data/processed/ to rerun.
X_train : (199020, 30)  dtype=float32  (normal only)
X_val   : (42893, 30)  dtype=float32
y_val   : fraud_rate=0.5735%  fraud_count=246
features: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
NaN check passed.


## 4. Verify GPU & Model Architecture

In [74]:
import torch, numpy as np
from pathlib import Path
import sys; sys.path.insert(0, '.')

device = (
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)
print(f'Device     : {device}')
if device == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
elif device == 'cpu':
    print('WARNING: CPU only — training will be slower.')

proc = Path('data/processed')
X_train = np.load(proc / 'X_train.npy')
y_val   = np.load(proc / 'y_val.npy')
print(f'train rows : {len(X_train):,}  (normal transactions only)')
print(f'val normal : {(y_val==0).sum():,}  |  val fraud: {(y_val==1).sum():,}')

from src.model import BetaVAE
from src.config import BETA, LATENT_DIM, ENCODER_DIMS, DECODER_DIMS, INPUT_DIM
model    = BetaVAE(input_dim=INPUT_DIM)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nBetaVAE architecture:')
print(f'  Encoder : {INPUT_DIM} → {" → ".join(str(d) for d in ENCODER_DIMS)} → μ/logσ² ({LATENT_DIM})')
print(f'  Decoder : {LATENT_DIM} → {" → ".join(str(d) for d in DECODER_DIMS)} → {INPUT_DIM}')
print(f'  β={BETA}  latent_dim={LATENT_DIM}  total_params={n_params:,}')

Device     : cpu
train rows : 199,020  (normal transactions only)
val normal : 42,647  |  val fraud: 246

BetaVAE architecture:
  Encoder : 30 → 32 → 16 → μ/logσ² (4)
  Decoder : 4 → 16 → 32 → 30
  β=0.005  latent_dim=4  total_params=3,462


## 5. Train the VAE

- Trains on **normal transactions only** (Class 0).
- Validation uses the full val set (normal + fraud) to compute anomaly scores.
- **Early stopping on Val AUPRC** (patience = 10) — AUPRC is the correct metric for 0.17% fraud imbalance.
- Best checkpoint saved to `models/best_vae.pth` only when AUPRC improves.
- Uncomment the `cfg.EXPERIMENTS_DIR` line to persist to Google Drive.

In [75]:
from google.colab import drive
drive.mount('/content/drive')

# Ensure the folder exists on your Drive
import os
os.makedirs('/content/drive/MyDrive/Fraud_VAE_Project', exist_ok=True)
print("Google Drive mounted and project folder ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted and project folder ready.


In [76]:
import sys, os, importlib
REPO_DIR = '/content/Deep-Fraud-VAE'

# Ensure we are in the repository root
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import src.config as cfg
# Persist experiments across Colab VM resets:
cfg.EXPERIMENTS_DIR = '/content/drive/MyDrive/Fraud_VAE_Project'

# Reload src modules so any config changes take effect
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

from src.train import train
exp_dir = train()   # returns Path to this run's experiment folder

print()
print('Experiment artefacts saved to:', exp_dir)
for fname in sorted(os.listdir(str(exp_dir))):
    size = os.path.getsize(os.path.join(str(exp_dir), fname))
    print(f'  {fname:<30} {size/1024:.1f} KB')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [  1/200] | Train Loss: 33.8325 | Val Loss: 28.1879 | Val AUROC: 0.9405 | Val AUPRC: 0.6306 | β: 0.0001


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [  2/200] | Train Loss: 28.1497 | Val Loss: 26.5006 | Val AUROC: 0.9442 | Val AUPRC: 0.6501 | β: 0.0002


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Epoch [ 30/200] | Train Loss: 25.1814 | Val Loss: 23.7114 | Val AUROC: 0.9466 | Val AUPRC: 0.6732 | β: 0.0030


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [ 31/200] | Train Loss: 25.1778 | Val Loss: 23.7306 | Val AUROC: 0.9470 | Val AUPRC: 0.6746 | β: 0.0031


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [ 32/200] | Train Loss: 25.1421 | Val Loss: 23.6984 | Val AUROC: 0.9459 | Val AUPRC: 0.6744 | β: 0.0032

Experiment artefacts saved to: /content/Deep-Fraud-VAE/experiments/exp_07
  anomaly_scores.png             46.1 KB
  best_model.pth                 69.9 KB
  config_backup.py               4.1 KB
  loss_history.csv               1.9 KB
  training_curves.png            89.3 KB


## 6. Training Curves

In [77]:
import pandas as pd
import matplotlib.pyplot as plt

log_csv = exp_dir / 'loss_history.csv'
log_df  = pd.read_csv(log_csv)
print(f'Epochs trained : {len(log_df)}')
print(log_df[['epoch', 'train_loss', 'val_loss', 'val_auroc', 'val_auprc']].tail(5).to_string(index=False))

best_ep = int(log_df.loc[log_df['val_auprc'].idxmax(), 'epoch'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
ax = axes[0]
ax.plot(log_df['epoch'], log_df['train_loss'], label='Train Loss', color='steelblue')
ax.plot(log_df['epoch'], log_df['val_loss'],   label='Val Loss',   color='crimson', linestyle='--')
ax.axvline(best_ep, color='orange', linewidth=1.2, linestyle=':', label=f'best ep {best_ep}')
ax.set_xlabel('Epoch'); ax.set_title('β-ELBO Loss'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# AUROC
ax = axes[1]
ax.plot(log_df['epoch'], log_df['val_auroc'], color='steelblue', label='Val AUROC')
ax.axvline(best_ep, color='orange', linewidth=1.2, linestyle=':')
ax.set_xlabel('Epoch'); ax.set_title('Val AUROC'); ax.set_ylim(0, 1)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# AUPRC
ax = axes[2]
ax.plot(log_df['epoch'], log_df['val_auprc'], color='crimson', label='Val AUPRC')
ax.axvline(best_ep, color='orange', linewidth=1.2, linestyle=':', label=f'best ep {best_ep}')
ax.set_xlabel('Epoch'); ax.set_title('Val AUPRC  (early-stopping metric)'); ax.set_ylim(0, 1)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

best_auprc = log_df['val_auprc'].max()
best_auroc = log_df.loc[log_df['val_auprc'].idxmax(), 'val_auroc']
plt.suptitle(
    f'β-VAE Training  (best epoch={best_ep}  AUPRC={best_auprc:.4f}  AUROC={best_auroc:.4f})',
    fontsize=12,
)
plt.tight_layout()
import os; os.makedirs('reports/figures', exist_ok=True)
plt.savefig('reports/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best epoch: {best_ep}  |  AUPRC={best_auprc:.4f}  |  AUROC={best_auroc:.4f}')

Epochs trained : 32
 epoch  train_loss  val_loss  val_auroc  val_auprc
    28   25.199092 23.759403   0.947333   0.675110
    29   25.158817 23.682238   0.947403   0.673311
    30   25.181414 23.711380   0.946619   0.673158
    31   25.177812 23.730621   0.946994   0.674569
    32   25.142084 23.698351   0.945929   0.674423
Best epoch: 12  |  AUPRC=0.6760  |  AUROC=0.9462


## 7. Anomaly Score Distribution

Reconstruction error (mean squared error per sample) is the anomaly score.

**Expected:** Fraud histogram shifted right of Normal.  
**If heavily overlapping:** lower `BETA` in `src/config.py` and retrain.

In [78]:
import torch, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score
from pathlib import Path
from src.model import BetaVAE
from src.config import DATA_PROC_DIR, INPUT_DIM

proc   = Path(DATA_PROC_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt  = torch.load('models/best_vae.pth', map_location=device)
model = BetaVAE(input_dim=INPUT_DIM).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Checkpoint: epoch={ckpt['epoch']}  AUPRC={ckpt['val_auprc']:.4f}  AUROC={ckpt['val_auroc']:.4f}")

X_val = torch.from_numpy(np.load(proc / 'X_val.npy').astype('float32')).to(device)
y_val = np.load(proc / 'y_val.npy')

with torch.no_grad():
    x_hat, _, _ = model(X_val)
    scores = ((X_val - x_hat) ** 2).mean(dim=1).cpu().numpy()

normal_s = scores[y_val == 0]
fraud_s  = scores[y_val == 1]
sep      = fraud_s.mean() / (normal_s.mean() + 1e-9)

auroc = roc_auc_score(y_val, scores)
auprc = average_precision_score(y_val, scores)
print(f'Normal mean={normal_s.mean():.4f}  p95={np.percentile(normal_s, 95):.4f}')
print(f'Fraud  mean={fraud_s.mean():.4f}  p95={np.percentile(fraud_s,  95):.4f}')
print(f'Separation : {sep:.2f}×')
print(f'AUROC      : {auroc:.4f}')
print(f'AUPRC      : {auprc:.4f}')

clip = float(np.percentile(scores, 99))
bins = np.linspace(0, clip, 80)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(normal_s.clip(max=clip), bins=bins, alpha=0.6, density=True,
        color='steelblue', label=f'Normal  (n={len(normal_s):,})')
ax.hist(fraud_s.clip(max=clip),  bins=bins, alpha=0.6, density=True,
        color='crimson',   label=f'Fraud   (n={len(fraud_s):,})')
ax.set_xlabel('Reconstruction Error (anomaly score)')
ax.set_ylabel('Density')
ax.set_title(
    f'Anomaly Score Distribution — Val Set\n'
    f'Separation={sep:.2f}×  |  AUROC={auroc:.4f}  |  AUPRC={auprc:.4f}'
)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/anomaly_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Checkpoint: epoch=12  AUPRC=0.6760  AUROC=0.9462
Normal mean=0.5792  p95=1.3571
Fraud  mean=6.9639  p95=14.7110
Separation : 12.02×
AUROC      : 0.9345
AUPRC      : 0.4948


In [79]:
import os
import shutil
from pathlib import Path

# Define paths
drive_root = '/content/drive/MyDrive/Fraud_VAE_Project'
local_experiments_root = '/content/Deep-Fraud-VAE/experiments'

print(f"Scanning local experiments in: {local_experiments_root}")

if os.path.exists(local_experiments_root):
    local_folders = [f for f in os.listdir(local_experiments_root) if os.path.isdir(os.path.join(local_experiments_root, f))]
    print(f"Found local folders: {local_folders}")

    for folder in local_folders:
        local_src = os.path.join(local_experiments_root, folder)
        drive_dst = os.path.join(drive_root, folder)

        if not os.path.exists(drive_dst):
            print(f"Copying {folder} to Drive...")
            shutil.copytree(local_src, drive_dst)
            print(f"Successfully saved {folder} to Drive.")
        else:
            # If it exists, sync files to ensure it's up to date
            print(f"Folder {folder} already exists on Drive. Checking for missing files...")
            for item in os.listdir(local_src):
                s = os.path.join(local_src, item)
                d = os.path.join(drive_dst, item)
                if not os.path.exists(d):
                    if os.path.isdir(s):
                        shutil.copytree(s, d)
                    else:
                        shutil.copy2(s, d)
                    print(f"  Added {item} to {folder}")
    print("\nSync complete. Your Drive should now mirror the folder structure (exp_01, exp_02, etc.).")
else:
    print("Error: Local experiments directory not found.")

Scanning local experiments in: /content/Deep-Fraud-VAE/experiments
Found local folders: ['exp_07', 'exp_03', 'exp_02', 'exp_01', 'exp_05', 'exp_06', 'exp_04']
Copying exp_07 to Drive...
Successfully saved exp_07 to Drive.
Folder exp_03 already exists on Drive. Checking for missing files...
Folder exp_02 already exists on Drive. Checking for missing files...
Folder exp_01 already exists on Drive. Checking for missing files...
Folder exp_05 already exists on Drive. Checking for missing files...
Folder exp_06 already exists on Drive. Checking for missing files...
Folder exp_04 already exists on Drive. Checking for missing files...

Sync complete. Your Drive should now mirror the folder structure (exp_01, exp_02, etc.).
